In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Curation peptide datasets from AMPDB v1

This notebook processes and curates multiple **toxic and cytolytic peptide datasets** extracted from **AMPDB v1**. Each dataset corresponds to a specific biological activity (e.g., cytolysis, cytotoxicity, hemolysis, ichthyotoxicity, insecticidal activity), and is handled independently to ensure clean, non-redundant sequence collections.

- **Toxic effect / endpoint:** cytolysis, cytolytic, cytotoxin, hemolytic and, ichthyotoxic.
- **Source:** AMPDB v1
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:
- **Loads raw TSV datasets** from AMPDB v1, including:
  - cytolysis
  - cytolytic
  - cytotoxin
  - hemolytic
  - ichthyotoxic
- **Standardizes column names and formats**, keeping only:
  - `sequence` (peptide sequence),
  - `label` (assigned as `1` for all datasets, since they represent positive activity classes).
- **Checks and resolves duplicated sequences within each dataset**:
  - unique sequences are preserved,
  - duplicated sequences with consistent labels are collapsed,
  - any unexpected inconsistencies are flagged as errors.
- **Concatenates curated subsets** into a global AMPDB v1 collection for metadata reporting.
- **Builds metadata** using the project-wide Excel description file and adds dataset-level quality-control statistics.
- **Exports per-activity curated datasets**, a global error list, and a JSON metadata file.

In [2]:
name_source = "AMPDB v1"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_cytolysis = pd.read_csv(f"{PATH_INPUT}/{name_source}/Cytolysis dataset.tsv", sep="\t")
df_cytolytic = pd.read_csv(f"{PATH_INPUT}/{name_source}/Cytolytic dataset.tsv", sep="\t")
df_cytotoxin = pd.read_csv(f"{PATH_INPUT}/{name_source}/Cytotoxin dataset.tsv", sep="\t")
df_hemolytic = pd.read_csv(f"{PATH_INPUT}/{name_source}/Hemolytic dataset.tsv", sep="\t")
df_ichthyotoxic = pd.read_csv(f"{PATH_INPUT}/{name_source}/Ichthyotoxic dataset.tsv", sep="\t")

In [4]:
df_cytolysis = (
    df_cytolysis
    .rename(columns={" Sequence": "sequence"})
    .assign(label=1)
    [["sequence", "label"]]
)
df_cytolysis.shape

(2329, 2)

In [5]:
df_cytolytic = (
    df_cytolytic
    .rename(columns={" Sequence": "sequence"})
    .assign(label=1)
    [["sequence", "label"]]
)
df_cytolytic.shape

(52, 2)

In [6]:
df_cytotoxin = (
    df_cytotoxin
    .rename(columns={" Sequence": "sequence"})
    .assign(label=1)
    [["sequence", "label"]]
)
df_cytotoxin.shape

(2727, 2)

In [7]:
df_hemolytic = (
    df_hemolytic
    .rename(columns={" Sequence": "sequence"})
    .assign(label=1)
    [["sequence", "label"]]
)
df_hemolytic.shape

(867, 2)

In [8]:
df_ichthyotoxic = (
    df_ichthyotoxic
    .rename(columns={" Sequence": "sequence"})
    .assign(label=1)
    [["sequence", "label"]]
)
df_ichthyotoxic.shape

(7, 2)

- Checking duplicates

In [9]:
df_remove_duplicated_cytolysis, df_errors_cytolysis, df_unique_cytolysis = processing_duplicated(df_cytolysis, group_seq="sequence", sort_key="label")

In [10]:
df_remove_duplicated_cytolytic, df_errors_cytolytic, df_unique_cytolytic = processing_duplicated(df_cytolytic, group_seq="sequence", sort_key="label")

In [11]:
df_remove_duplicated_cytotoxin, df_errors_cytotoxin, df_unique_cytotoxin = processing_duplicated(df_cytotoxin, group_seq="sequence", sort_key="label")

In [12]:
df_remove_duplicated_hemolytic, df_errors_hemolytic, df_unique_hemolytic = processing_duplicated(df_hemolytic, group_seq="sequence", sort_key="label")

In [13]:
df_remove_duplicated_ichthyotoxic, df_errors_ichthyotoxic, df_unique_ichthyotoxic = processing_duplicated(df_ichthyotoxic, group_seq="sequence", sort_key="label")

In [14]:
df_full_cytolysis = pd.concat([df_unique_cytolysis, df_remove_duplicated_cytolysis])
df_full_cytolytic = pd.concat([df_unique_cytolytic, df_remove_duplicated_cytolytic])
df_full_cytotoxin = pd.concat([df_unique_cytotoxin, df_remove_duplicated_cytotoxin])
df_full_hemolytic = pd.concat([df_unique_hemolytic, df_remove_duplicated_hemolytic])
df_full_ichthyotoxic = pd.concat([df_unique_ichthyotoxic, df_remove_duplicated_ichthyotoxic])
df_full = pd.concat([df_full_cytolysis, df_full_cytolytic, df_full_cytotoxin, 
                     df_full_hemolytic, df_full_ichthyotoxic])
df_errors = pd.concat([df_errors_cytolysis, df_errors_cytolytic, df_errors_cytotoxin, 
                       df_errors_hemolytic, df_errors_ichthyotoxic])

In [15]:
df_errors.shape

(0, 1)

- Working with metada

In [16]:
df_metada = read_metadata_multiple("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada) 

In [17]:
raw_total_sequences = (len(df_cytolysis) + len(df_cytolytic) + len(df_hemolytic) + len(df_cytotoxin) + len(df_ichthyotoxic))

In [18]:
dict_metadata.update({
    "number_of_raw_sequences": int(raw_total_sequences),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Database',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2023,
 'last update date': '2023-05-03 00:00:00;2023-05-08 00:00:00;2023-05-09 00:00:00;2023-05-10 00:00:00;2023-05-15 00:00:00;2023-05-17 00:00:00;2023-05-18 00:00:00;2023-05-22 00:00:00;2023-05-26 00:00:00',
 'download date': Timestamp('2025-08-01 00:00:00'),
 'file format': 'tsv',
 'peptide property': 'blood coagulation cascade inhibiting toxin peptides, toxic;cytolytic, toxic;cytotoxic, toxic;hemolytic, toxic;ichthyotoxic peptides, toxic;insecticidal, toxic;platelet aggregation inhibiting toxin, toxins, toxic;toxins, toxic',
 'dataset information': 'Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'https://bblserver.org.in/ampdb/',
 'publication': 'https://www.nature.com/articles/s41598-023-45016-3',
 'number_of_raw_sequences': 5982,
 'number_of_sequences_retained': 5230,
 'number_of_positive_seq

- Exporting data

In [19]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [20]:
df_full_cytolysis.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_cytolysis_dataset.csv", index=False)
df_full_cytolytic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_cytolytic_dataset.csv", index=False)
df_full_cytotoxin.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_cytotoxic_dataset.csv", index=False)
df_full_hemolytic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_ichthyotoxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_ichthyotoxic_dataset.csv", index=False)

df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)